In [1]:
from pandas import ExcelFile
import geopandas as gpd
from pandas import read_csv
from pandas import DataFrame
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from PaddockTS.Data.environmental import download_environmental_data
from PaddockTS.get_outputs import get_outputs
from PaddockTS.query import Query
from datetime import date

2025-09-10 23:25:35.259154: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-10 23:25:35.318429: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-10 23:25:44.739752: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [22]:
import time
from time import perf_counter

In [3]:
df = read_csv('/g/data/xe2/ya6227/NVTAnalysis/data/selected_10_Hyola_Blazer_TT.csv')

In [4]:
for _, row in df.iterrows():
    query = Query(
        lat=row['Trial GPS Lat'],
        lon=row['Trial GPS Long'],
        collections=['ga_s2am_ard_3', 'ga_s2bm_ard_3'],
        buffer=0.01,
        bands=[
            'nbart_blue',
            'nbart_green',
            'nbart_red',
            'nbart_red_edge_1',
            'nbart_red_edge_2',
            'nbart_red_edge_3',
            'nbart_nir_1',
            'nbart_nir_2',
            'nbart_swir_2',
            'nbart_swir_3'
        ],
        start_time=date.fromisoformat(row['SowingDate']),
        end_time=date.fromisoformat(row['HarvestDate']),
        out_dir='/g/data/xe2/ya6227/NVTAnalysis/data/DAESim',
        tmp_dir='/g/data/xe2/ya6227/NVTAnalysis/data/DAESim'
    )
    break

In [5]:
from daesim2_analysis.experiment import Experiment
from daesim2_analysis.parameters import Parameters
from daesim2_analysis.daesim_config import DAESIMConfig
from daesim2_analysis.run import *

In [13]:
df_forcing = f'{query.stub_tmp_dir}/environmental/{query.stub}_DAESim_forcing.csv'

In [14]:
df_forcing

'/g/data/xe2/ya6227/NVTAnalysis/data/DAESim/91093d349d7b3926a7507f10d7c2f533fa335e4288add43fc04aa35145d9986c/environmental/91093d349d7b3926a7507f10d7c2f533fa335e4288add43fc04aa35145d9986c_DAESim_forcing.csv'

In [15]:
parameters = Parameters(
        paths           = ["PlantCH2O.CanopyGasExchange.Leaf", "PlantCH2O.CanopyGasExchange.Leaf", "PlantCH2O", "PlantCH2O", "PlantCH2O", "PlantCH2O", "PlantCH2O", "PlantDev", "PlantDev", "", "", "", ""],
        modules         = ["Leaf", "Leaf", "PlantCH2O", "PlantCH2O", "PlantCH2O", "PlantCH2O", "PlantCH2O", "PlantDev", "PlantDev", "", "", "", ""],
        names           = ["Vcmax_opt", "g1", "SLA", "maxLAI", "ksr_coeff", "Psi_f", "sf", "gdd_requirements", "gdd_requirements", "GY_FE", "GY_SDW_50", "CI", "d_r_max"],
        units           = ["mol CO2 m-2 s-1", "kPa^0.5", "m2 g d.wt-1", "m2 m-2", "g d.wt-1 m-1", "MPa", "MPa-1", "deg C d", "deg C d", "thsnd grains g d.wt spike-1", "g d.wt m-2", "-", "m"],
        init            = [60e-6, 3, 0.03, 6, 1000, -3.5, 3.5, 900, 650, 0.1, 100, 0.75, 0.5],
        min             = [30e-6, 1, 0.015, 5, 300, -8.0, 1.5, 600, 350, 0.08, 80, 0.5, 0.15],
        max             = [120e-6, 6, 0.035, 7, 5000, -1.0, 7.0, 1800, 700, 0.21, 150, 1.0, 0.66],
        phase_specific  = [False, False, False, False, False, False, False, True, True, False, False, False, False],
        phase           = [None, None, None, None, None, None, None, "vegetative", "grainfill", None, None, None, None]
)

In [16]:
daesim_config = DAESIMConfig.from_json_dict("/g/data/xe2/ya6227/daesim2-analysis/daesim_configs/DAESIM1.json") 

In [17]:
experiment = Experiment(
	xsite="TestSite",
	crop_type='Canola',
	CLatDeg=query.lat,
	CLonDeg=query.lon,
	df_forcing=df_forcing,
    parameters=parameters,
    daesim_config=daesim_config,
    sowing_dates=[query.start_time],
    harvest_dates=[query.end_time],
)

In [18]:
parameters: Parameters = experiment.parameters
param_values = parameters.sample(experiment.n_samples)

# %%
# Call the function that updates parameters, runs the model and returns selected outputs


In [23]:
start = perf_counter()
model_output = update_and_run_model(
    parameters.init, 
    experiment.PlantX,
    experiment.input_data,
    parameters.df,
    parameters.problem
)
print(perf_counter() - start)

/g/data/xe2/ya6227/miniconda3/envs/PaddockTSEnv/lib/python3.10/site-packages/daesim/plantcarbonwater.py:459: RuntimeWarning: divide by zero encountered in divide
  r_ws = (airP/(self.Site.R_w_mol*(leafTempC+273.15)))/gsw    ## converts stomatal conductance (mol H2O m-2 s-1) to stomatal resistance (s m-1)  (see Nobel (2009) Section 8.1F) TODO: Check this and cross-check it with calculations of stomatal resistance in leafgasexchange modules and perhaps also SCOPE model code


29.13782171299681


In [24]:
29 * 1e+6

29000000.0